In [1]:
#python
import os

# 작업 디렉토리 확인
print(f"현재 위치: {os.getcwd()}")

현재 위치: c:\Users\하노\Jupyter


In [2]:
# python
# 가상환경이 생성되었는지 확인
import os
print(os.path.exists(".venv"))       # True가 출력되어야 합니다
print(os.listdir(".venv"))           # ['bin', 'include', 'lib', 'pyvenv.cfg'] 등

True
['etc', 'Include', 'Lib', 'pyvenv.cfg', 'Scripts', 'share']


In [3]:
# bash
# 가상환경의 pip으로 ipykernel 설치
!.venv\Scripts\pip install ipykernel

In [4]:
# bash
# 주피터 커널로 등록
!.venv\Scripts\python -m ipykernel install --user --name=model-serving --display-name="Model Serving"

Installed kernelspec model-serving in C:\Users\하노\AppData\Roaming\jupyter\kernels\model-serving


In [2]:
#python
# 커널을 "Model Serving"으로 전환한 뒤 실행합니다
import sys
print(f"Python 경로: {sys.executable}")
# 출력에 .venv가 포함되어 있으면 성공입니다
# 예: /Users/yourname/model-serving-course/.venv/bin/python

Python 경로: c:\Users\하노\Jupyter\.venv\Scripts\python.exe


In [3]:
%%writefile requirements.txt
# ===== Core =====
torch>=2.1.0,<2.5.0
torchvision>=0.16.0,<0.20.0

# ===== API =====
fastapi==0.115.0
uvicorn[standard]==0.30.0
pydantic>=2.0.0,<3.0.0

# ===== Frontend =====
streamlit==1.38.0

# ===== Utilities =====
requests>=2.31.0,<3.0.0
pillow>=10.0.0
python-multipart>=0.0.6

Overwriting requirements.txt


In [4]:
!pip install -r requirements.txt

In [5]:
import torch
import fastapi
import streamlit

print(f"PyTorch:   {torch.__version__}")
print(f"FastAPI:   {fastapi.__version__}")
print(f"Streamlit: {streamlit.__version__}")

PyTorch:   2.4.1+cpu
FastAPI:   0.115.0
Streamlit: 1.38.0


In [6]:
import os

# 폴더 구조 생성
folders = [
    "models",          # 저장된 모델 파일 (.pth, .onnx 등)
    "app",             # FastAPI 애플리케이션 코드
    "frontend",        # Streamlit 프론트엔드 코드
    "notebooks",       # 주피터 노트북 (지금 이 파일)
    "data",            # 샘플 데이터
    "tests",           # 테스트 코드
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"✅ {folder}/ 생성 완료")

✅ models/ 생성 완료
✅ app/ 생성 완료
✅ frontend/ 생성 완료
✅ notebooks/ 생성 완료
✅ data/ 생성 완료
✅ tests/ 생성 완료


In [7]:
import json

# Python dict → JSON 문자열 (직렬화)
data = {
    "text": "이 영화 정말 재밌다",
    "return_probabilities": True,
    "max_length": None
}

json_string = json.dumps(data, ensure_ascii=False, indent=2)
print("=== Python → JSON ===")
print(json_string)
print(f"타입: {type(json_string)}")  #

=== Python → JSON ===
{
  "text": "이 영화 정말 재밌다",
  "return_probabilities": true,
  "max_length": null
}
타입: <class 'str'>


In [8]:
# JSON 문자열 → Python dict (역직렬화)
parsed = json.loads(json_string)
print("\n=== JSON → Python ===")
print(parsed)
print(f"타입: {type(parsed)}")       #
print(f"텍스트: {parsed['text']}")   # 이 영화 정말 재밌다


=== JSON → Python ===
{'text': '이 영화 정말 재밌다', 'return_probabilities': True, 'max_length': None}
타입: <class 'dict'>
텍스트: 이 영화 정말 재밌다


In [9]:
import requests

# JSONPlaceholder: 테스트용 공개 REST API
response = requests.get("https://jsonplaceholder.typicode.com/posts/1")

print(f"상태 코드: {response.status_code}")   # 200
print(f"응답 타입: {type(response.json())}")   #
print(f"응답 내용:")
print(json.dumps(response.json(), indent=2))

상태 코드: 200
응답 타입: <class 'dict'>
응답 내용:
{
  "userId": 1,
  "id": 1,
  "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
  "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
}


In [10]:
# POST 요청: 새로운 데이터를 전송합니다
response = requests.post(
    "https://jsonplaceholder.typicode.com/posts",
    json={                                # json= 을 사용하면 자동으로 JSON 변환 + 헤더 설정
        "title": "모델 배포 테스트",
        "body": "FastAPI로 모델을 서빙합니다",
        "userId": 1
    }
)

print(f"상태 코드: {response.status_code}")   # 201 (Created)
print(f"응답 내용:")
print(json.dumps(response.json(), ensure_ascii=False, indent=2))

상태 코드: 201
응답 내용:
{
  "title": "모델 배포 테스트",
  "body": "FastAPI로 모델을 서빙합니다",
  "userId": 1,
  "id": 101
}


In [11]:
# 존재하지 않는 리소스에 GET 요청
response = requests.get("https://jsonplaceholder.typicode.com/posts/99999")
print(f"상태 코드: {response.status_code}")   # 404 (Not Found)
print(f"응답 내용: {response.json()}")         # {}

상태 코드: 404
응답 내용: {}


In [12]:
# 잘못된 URL로 요청
try:
    response = requests.get("https://jsonplaceholder.typicode.com/없는경로")
    print(f"상태 코드: {response.status_code}")   # 404
except requests.exceptions.RequestException as e:
    print(f"요청 실패: {e}")

상태 코드: 404


In [13]:
import torch
import torch.nn as nn

In [14]:
class SimpleClassifier(nn.Module):
    """
    간단한 이미지 분류 모델
    - 입력: 1x28x28 (MNIST와 동일한 크기)
    - 출력: 10개 클래스에 대한 확률
    """
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [15]:
# 모델 인스턴스 생성
model = SimpleClassifier(num_classes=10)

# 더미 입력으로 동작 확인
dummy_input = torch.randn(1, 1, 28, 28)  # (batch=1, channels=1, height=28, width=28)
output = model(dummy_input)

print(f"모델 구조:\n{model}\n")
print(f"입력 크기: {dummy_input.shape}")
print(f"출력 크기: {output.shape}")          # torch.Size([1, 10])
print(f"출력 값:   {output.detach()}")

모델 구조:
SimpleClassifier(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)

입력 크기: torch.Size([1, 1, 28, 28])
출력 크기: torch.Size([1, 10])
출력 값:   tensor([[ 0.1400,  0.1170, -0.0738, -0.1484, -0.0909, -0.0603,  0.0309, -0.0888,
          0.0811, -0.2277]])


In [16]:
import os

# 모델 저장 폴더 확인
os.makedirs("models", exist_ok=True)

In [17]:
# state_dict: 모델의 가중치(파라미터)만 딕셔너리 형태로 저장합니다

# 1. 원본 모델을 평가 모드로 전환 (매우 중요!)
model.eval()

# 2. 원본 모델에서 출력값 생성
with torch.no_grad():
    output = model(dummy_input)

# 3. 모델 저장
torch.save(model.state_dict(), "models/model_state_dict.pth")

# 파일 크기 확인
file_size = os.path.getsize("models/model_state_dict.pth")
print(f"저장 완료: models/model_state_dict.pth")
print(f"파일 크기: {file_size / 1024:.1f} KB")

저장 완료: models/model_state_dict.pth
파일 크기: 1650.5 KB


In [18]:
# 저장된 내용 확인: 어떤 키들이 들어 있는지 살펴봅니다
state_dict = torch.load("models/model_state_dict.pth", weights_only=True)

print("저장된 키 목록:")
for key, tensor in state_dict.items():
    print(f"  {key:40s} → {tensor.shape}")


저장된 키 목록:
  features.0.weight                        → torch.Size([32, 1, 3, 3])
  features.0.bias                          → torch.Size([32])
  features.3.weight                        → torch.Size([64, 32, 3, 3])
  features.3.bias                          → torch.Size([64])
  classifier.1.weight                      → torch.Size([128, 3136])
  classifier.1.bias                        → torch.Size([128])
  classifier.4.weight                      → torch.Size([10, 128])
  classifier.4.bias                        → torch.Size([10])


In [19]:
# 불러올 때는 반드시 동일한 모델 클래스가 필요합니다
# 1. 원본 모델을 평가 모드로 전환 (매우 중요!)
model.eval()

# 2. 원본 모델에서 출력값 생성
with torch.no_grad():
    output = model(dummy_input)

# 3. 모델 저장
torch.save(model.state_dict(), "models/model_state_dict.pth")

# --- 여기서부터 불러오기 ---

# 4. 새 모델 선언 및 가중치 로드
loaded_model = SimpleClassifier(num_classes=10)
loaded_model.load_state_dict(torch.load("models/model_state_dict.pth", weights_only=True))

# 5. 불러온 모델도 평가 모드로 전환
loaded_model.eval()

# 6. 복원된 모델에서 출력값 생성
with torch.no_grad():
    loaded_output = loaded_model(dummy_input)

In [20]:
# 동일한 입력에 대해 동일한 출력이 나오는지 확인
with torch.no_grad():
    loaded_output = loaded_model(dummy_input)

print(f"원본 출력:  {output.detach()}")
print(f"복원 출력:  {loaded_output}")
print(f"동일 여부:  {torch.allclose(output.detach(), loaded_output)}")  # True

원본 출력:  tensor([[ 0.1189,  0.0815, -0.0031, -0.1263, -0.0466, -0.0517, -0.0903, -0.0793,
         -0.0073, -0.0988]])
복원 출력:  tensor([[ 0.1189,  0.0815, -0.0031, -0.1263, -0.0466, -0.0517, -0.0903, -0.0793,
         -0.0073, -0.0988]])
동일 여부:  True


In [21]:
# eval 모드로 전환 (Dropout, BatchNorm 등의 동작이 달라지므로 필수)
model.eval()

# trace: 더미 입력을 한 번 통과시켜서 모델의 연산 그래프를 기록합니다
traced_model = torch.jit.trace(model, dummy_input)

# 저장
traced_model.save("models/model_traced.pt")

file_size = os.path.getsize("models/model_traced.pt")
print(f"저장 완료: models/model_traced.pt")
print(f"파일 크기: {file_size / 1024:.1f} KB")


저장 완료: models/model_traced.pt
파일 크기: 1673.3 KB


In [22]:
# 핵심: 모델 클래스 정의가 필요 없습니다!
loaded_traced = torch.jit.load("models/model_traced.pt")

with torch.no_grad():
    traced_output = loaded_traced(dummy_input)

print(f"원본 출력:       {output.detach()}")
print(f"TorchScript 출력: {traced_output}")
print(f"동일 여부:        {torch.allclose(output.detach(), traced_output)}")  # True

원본 출력:       tensor([[ 0.1189,  0.0815, -0.0031, -0.1263, -0.0466, -0.0517, -0.0903, -0.0793,
         -0.0073, -0.0988]])
TorchScript 출력: tensor([[ 0.1189,  0.0815, -0.0031, -0.1263, -0.0466, -0.0517, -0.0903, -0.0793,
         -0.0073, -0.0988]])
동일 여부:        True


In [23]:
# 방법 A: torch.jit.trace
# - 더미 입력을 실제로 통과시켜서 연산 경로를 기록합니다.
# - 장점: 대부분의 모델에서 잘 동작합니다.
# - 단점: 입력에 따라 분기(if/else)하는 로직은 기록되지 않습니다.
traced = torch.jit.trace(model, dummy_input)

# 방법 B: torch.jit.script
# - Python 코드를 직접 분석하여 TorchScript IR로 컴파일합니다.
# - 장점: if/else, for 루프 등 동적 로직도 변환됩니다.
# - 단점: Python 문법 중 지원되지 않는 것이 있어 에러가 발생할 수 있습니다.
scripted = torch.jit.script(model)

In [24]:
# 두 방식 모두 동일한 결과를 내는지 확인
with torch.no_grad():
    trace_out = traced(dummy_input)
    script_out = scripted(dummy_input)

print(f"trace 출력:  {trace_out}")
print(f"script 출력: {script_out}")
print(f"동일 여부:   {torch.allclose(trace_out, script_out)}")  # True

trace 출력:  tensor([[ 0.1189,  0.0815, -0.0031, -0.1263, -0.0466, -0.0517, -0.0903, -0.0793,
         -0.0073, -0.0988]])
script 출력: tensor([[ 0.1189,  0.0815, -0.0031, -0.1263, -0.0466, -0.0517, -0.0903, -0.0793,
         -0.0073, -0.0988]])
동일 여부:   True


In [25]:
!pip install onnx onnxscript onnxruntime

  Using cached onnx-1.22.0-cp312-abi3-win_amd64.whl.metadata (8.5 kB)
  Using cached onnxscript-0.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached onnxruntime-1.28.0-cp312-cp312-win_amd64.whl.metadata (5.7 kB)
  Using cached ml_dtypes-0.5.4-cp312-cp312-win_amd64.whl.metadata (9.2 kB)
  Using cached onnx_ir-1.0.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
Using cached onnx-1.22.0-cp312-abi3-win_amd64.whl (17.2 MB)
Using cached onnxscript-0.7.1-py3-none-any.whl (721 kB)
Using cached onnx_ir-1.0.0-py3-none-any.whl (185 kB)
Using cached onnxruntime-1.28.0-cp312-cp312-win_amd64.whl (13.8 MB)
Using cached ml_dtypes-0.5.4-cp312-cp312-win_amd64.whl (212 kB)
Using cached flatbuffers-25.12.19-py2.py3-none-any.whl (26 kB)

   ------ --------------------------------- 1/6 [onnxruntime]
   ------ --------------------------------- 1/6 [onnxruntime]
   ------ --------------------------------- 1/6 [onnxruntime]
   ------ -----------

In [26]:
import torch, onnx

model.eval()

torch.onnx.export(
    model,                                    # 변환할 모델
    dummy_input,                              # 더미 입력 (모델 트레이싱에 사용)
    "models/model.onnx",                      # 저장 경로
    export_params=True,                       # 가중치를 파일에 포함
    opset_version=17,                         # ONNX 연산자 버전
    input_names=["image"],                    # 입력 텐서 이름
    output_names=["prediction"],              # 출력 텐서 이름
    dynamic_axes={                            # 가변 크기 축 지정
        "image": {0: "batch_size"},           # batch 크기를 동적으로
        "prediction": {0: "batch_size"},
    }
)

In [27]:
import onnx

# 모델 구조 검증
onnx_model = onnx.load("models/model.onnx")
onnx.checker.check_model(onnx_model)
print("✅ ONNX 모델 검증 통과")

# 모델 정보 확인
print(f"\n입력:")
for inp in onnx_model.graph.input:
    print(f"  이름: {inp.name}")
    shape = [d.dim_param or d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"  크기: {shape}")

print(f"\n출력:")
for out in onnx_model.graph.output:
    print(f"  이름: {out.name}")
    shape = [d.dim_param or d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"  크기: {shape}")

✅ ONNX 모델 검증 통과

입력:
  이름: image
  크기: ['batch_size', 1, 28, 28]

출력:
  이름: prediction
  크기: ['batch_size', 10]


In [28]:
import onnxruntime as ort
import numpy as np

# ONNX Runtime 세션 생성
session = ort.InferenceSession("models/model.onnx")

# 입력 데이터 준비 (NumPy 배열로 변환)
input_data = dummy_input.numpy()

# 추론 실행
onnx_output = session.run(
    output_names=["prediction"],
    input_feed={"image": input_data}
)

In [29]:
print(f"PyTorch 출력:       {output.detach().numpy()}")
print(f"ONNX Runtime 출력:  {onnx_output[0]}")
print(f"동일 여부 (오차 허용): {np.allclose(output.detach().numpy(), onnx_output[0], atol=1e-5)}")

PyTorch 출력:       [[ 0.11890037  0.08150814 -0.00307651 -0.12631321 -0.04664617 -0.05172645
  -0.09027895 -0.07934543 -0.00725777 -0.09877293]]
ONNX Runtime 출력:  [[ 0.1189004   0.08150814 -0.0030766  -0.12631315 -0.04664626 -0.05172651
  -0.09027901 -0.07934536 -0.00725774 -0.09877295]]
동일 여부 (오차 허용): True


In [30]:
# 저장된 파일 크기 비교
files = {
    "state_dict (.pth)": "models/model_state_dict.pth",
    "TorchScript (.pt)":  "models/model_traced.pt",
    "ONNX (.onnx)":       "models/model.onnx",
}

print(f"{'방식':<25} {'파일 크기':>10}")
print("-" * 37)
for name, path in files.items():
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"{name:<25} {size_kb:>8.1f} KB")

방식                             파일 크기
-------------------------------------
state_dict (.pth)           1650.5 KB
TorchScript (.pt)           1673.3 KB
ONNX (.onnx)                1649.1 KB


In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [32]:
# ===== 모델 정의 (섹션 4와 동일) =====
class SimpleClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [33]:
# ===== 하이퍼파라미터 =====
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 3            # 실습용이므로 3 에포크만 학습합니다

# ===== 디바이스 설정 =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

사용 디바이스: cpu


In [34]:
# ===== 데이터 준비 =====
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))   # MNIST 평균/표준편차
])

train_dataset = datasets.MNIST(
    root="data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root="data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"학습 데이터: {len(train_dataset):,}장")
print(f"테스트 데이터: {len(test_dataset):,}장")

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data\MNIST\raw\train-images-idx3-ubyte.gz to data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data\MNIST\raw\train-labels-idx1-ubyte.gz to data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data\MNIST\raw\t10k-images-idx3-ubyte.gz to data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%

Extracting data\MNIST\raw\t10k-labels-idx1-ubyte.gz to data\MNIST\raw

학습 데이터: 60,000장
테스트 데이터: 10,000장


In [35]:
model = SimpleClassifier(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [36]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # 200 배치마다 진행 상황 출력
        if (batch_idx + 1) % 200 == 0:
            print(f"  Epoch {epoch} [{batch_idx+1}/{len(train_loader)}] "
                  f"Loss: {running_loss/(batch_idx+1):.4f} "
                  f"Acc: {100.*correct/total:.1f}%")

    # 에포크 종료 시 요약
    train_acc = 100. * correct / total
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch}/{EPOCHS} 완료 — Loss: {avg_loss:.4f}, Acc: {train_acc:.1f}%\n")

  Epoch 1 [200/938] Loss: 0.4910 Acc: 84.6%
  Epoch 1 [400/938] Loss: 0.3349 Acc: 89.6%
  Epoch 1 [600/938] Loss: 0.2711 Acc: 91.7%
  Epoch 1 [800/938] Loss: 0.2327 Acc: 92.9%
Epoch 1/3 완료 — Loss: 0.2152, Acc: 93.4%

  Epoch 2 [200/938] Loss: 0.0914 Acc: 97.3%
  Epoch 2 [400/938] Loss: 0.0882 Acc: 97.4%
  Epoch 2 [600/938] Loss: 0.0861 Acc: 97.5%
  Epoch 2 [800/938] Loss: 0.0864 Acc: 97.5%
Epoch 2/3 완료 — Loss: 0.0844, Acc: 97.6%

  Epoch 3 [200/938] Loss: 0.0683 Acc: 98.0%
  Epoch 3 [400/938] Loss: 0.0660 Acc: 98.0%
  Epoch 3 [600/938] Loss: 0.0673 Acc: 98.0%
  Epoch 3 [800/938] Loss: 0.0659 Acc: 98.0%
Epoch 3/3 완료 — Loss: 0.0647, Acc: 98.1%



In [37]:
import os
os.makedirs("models", exist_ok=True)

# 모델을 CPU로 이동 (배포 환경에서는 GPU가 없을 수 있으므로)
model_cpu = model.cpu()
model_cpu.eval()

# 추론 비교용 테스트 입력
test_input = test_dataset[0][0].unsqueeze(0)   # 첫 번째 테스트 이미지
test_label = test_dataset[0][1]                 # 정답 레이블

print(f"테스트 입력 크기: {test_input.shape}")
print(f"정답 레이블: {test_label}")

테스트 입력 크기: torch.Size([1, 1, 28, 28])
정답 레이블: 7


In [38]:
# 저장 전 원본 모델의 추론 결과를 기록해 둡니다
with torch.no_grad():
    original_output = model_cpu(test_input)
    original_pred = original_output.argmax(dim=1).item()
    original_conf = torch.softmax(original_output, dim=1).max().item()

print(f"원본 모델 예측: {original_pred} (확신도: {original_conf:.4f})")
print(f"정답:          {test_label}")
print(f"정답 여부:      {'✅ 맞음' if original_pred == test_label else '❌ 틀림'}")

원본 모델 예측: 7 (확신도: 1.0000)
정답:          7
정답 여부:      ✅ 맞음


In [39]:
# state_dict 저장
torch.save(model_cpu.state_dict(), "models/mnist_state_dict.pth")
print(f"✅ state_dict 저장 완료: {os.path.getsize('models/mnist_state_dict.pth') / 1024:.1f} KB")

✅ state_dict 저장 완료: 1650.5 KB


In [40]:
# TorchScript 변환 및 저장
traced_model = torch.jit.trace(model_cpu, test_input)
traced_model.save("models/mnist_traced.pt")
print(f"✅ TorchScript 저장 완료: {os.path.getsize('models/mnist_traced.pt') / 1024:.1f} KB")

✅ TorchScript 저장 완료: 1674.2 KB


In [41]:
import onnxscript

In [42]:
# ONNX 변환 및 저장
torch.onnx.export(
    model_cpu,
    test_input,
    "models/mnist_model.onnx",
    export_params=True,
    opset_version=17,
    input_names=["image"],
    output_names=["prediction"],
    dynamic_axes={
        "image": {0: "batch_size"},
        "prediction": {0: "batch_size"},
    }
)
print(f"✅ ONNX 저장 완료: {os.path.getsize('models/mnist_model.onnx') / 1024:.1f} KB")

✅ ONNX 저장 완료: 1649.1 KB


In [43]:
# 저장 결과 요약
print("\n" + "=" * 50)
print("📁 models/ 폴더 내용")
print("=" * 50)
for fname in sorted(os.listdir("models")):
    fpath = os.path.join("models", fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:<30} {size_kb:>8.1f} KB")


📁 models/ 폴더 내용
  mnist_model.onnx                 1649.1 KB
  mnist_state_dict.pth             1650.5 KB
  mnist_traced.pt                  1674.2 KB
  model.onnx                       1649.1 KB
  model_state_dict.pth             1650.5 KB
  model_traced.pt                  1673.3 KB


In [44]:
# 클래스 정의가 반드시 있어야 합니다
loaded_sd = SimpleClassifier(num_classes=10)
loaded_sd.load_state_dict(
    torch.load("models/mnist_state_dict.pth", weights_only=True)
)
loaded_sd.eval()

with torch.no_grad():
    sd_output = loaded_sd(test_input)
    sd_pred = sd_output.argmax(dim=1).item()

print(f"[state_dict] 예측: {sd_pred}, 원본과 일치: {torch.allclose(original_output, sd_output)}")

[state_dict] 예측: 7, 원본과 일치: True


In [45]:
# 클래스 정의가 필요 없습니다
loaded_ts = torch.jit.load("models/mnist_traced.pt")

with torch.no_grad():
    ts_output = loaded_ts(test_input)
    ts_pred = ts_output.argmax(dim=1).item()

print(f"[TorchScript] 예측: {ts_pred}, 원본과 일치: {torch.allclose(original_output, ts_output)}")

[TorchScript] 예측: 7, 원본과 일치: True


In [46]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession("models/mnist_model.onnx")
onnx_output = session.run(
    ["prediction"],
    {"image": test_input.numpy()}
)

onnx_pred = np.argmax(onnx_output[0], axis=1)[0]
match = np.allclose(original_output.numpy(), onnx_output[0], atol=1e-5)

print(f"[ONNX]        예측: {onnx_pred}, 원본과 일치 (오차 허용): {match}")

[ONNX]        예측: 7, 원본과 일치 (오차 허용): True


In [47]:
print("\n" + "=" * 60)
print("📊 직렬화 검증 결과 요약")
print("=" * 60)
print(f"  정답 레이블:        {test_label}")
print(f"  원본 모델 예측:     {original_pred}")
print(f"  state_dict 예측:   {sd_pred}  {'✅' if sd_pred == original_pred else '❌'}")
print(f"  TorchScript 예측:  {ts_pred}  {'✅' if ts_pred == original_pred else '❌'}")
print(f"  ONNX 예측:         {onnx_pred}  {'✅' if onnx_pred == original_pred else '❌'}")
print("=" * 60)

if all(p == original_pred for p in [sd_pred, ts_pred, onnx_pred]):
    print("\n🎉 세 가지 방식 모두 원본과 동일한 결과를 반환합니다.")
    print("   모델을 안전하게 직렬화하고 복원할 수 있다는 것이 검증되었습니다.")


📊 직렬화 검증 결과 요약
  정답 레이블:        7
  원본 모델 예측:     7
  state_dict 예측:   7  ✅
  TorchScript 예측:  7  ✅
  ONNX 예측:         7  ✅

🎉 세 가지 방식 모두 원본과 동일한 결과를 반환합니다.
   모델을 안전하게 직렬화하고 복원할 수 있다는 것이 검증되었습니다.


In [48]:
# 테스트 데이터에서 8장을 배치로 묶습니다
batch_images = torch.stack([test_dataset[i][0] for i in range(8)])
batch_labels = [test_dataset[i][1] for i in range(8)]

print(f"배치 입력 크기: {batch_images.shape}")  # torch.Size([8, 1, 28, 28])

배치 입력 크기: torch.Size([8, 1, 28, 28])


In [49]:
# 세 가지 방식으로 배치 추론
with torch.no_grad():
    sd_batch = loaded_sd(batch_images).argmax(dim=1).tolist()
    ts_batch = loaded_ts(batch_images).argmax(dim=1).tolist()

onnx_batch_out = session.run(["prediction"], {"image": batch_images.numpy()})
onnx_batch = np.argmax(onnx_batch_out[0], axis=1).tolist()

# 결과 비교
print(f"\n{'이미지':<8} {'정답':<6} {'state_dict':<12} {'TorchScript':<13} {'ONNX':<8}")
print("-" * 50)
for i in range(8):
    match = "✅" if sd_batch[i] == ts_batch[i] == onnx_batch[i] == batch_labels[i] else "❌"
    print(f"  #{i:<5} {batch_labels[i]:<6} {sd_batch[i]:<12} {ts_batch[i]:<13} {onnx_batch[i]:<8} {match}")


이미지      정답     state_dict   TorchScript   ONNX    
--------------------------------------------------
  #0     7      7            7             7        ✅
  #1     2      2            2             2        ✅
  #2     1      1            1             1        ✅
  #3     0      0            0             0        ✅
  #4     4      4            4             4        ✅
  #5     1      1            1             1        ✅
  #6     4      4            4             4        ✅
  #7     9      9            9             9        ✅


In [50]:
%%writefile app/model_utils.py
"""
모델 로드 및 추론 유틸리티
FastAPI 엔드포인트가 이 모듈을 import하여 사용합니다.
"""


import torch
import torch.nn as nn
from torchvision import transforms


# ===== 모델 정의 =====
class SimpleClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# ===== 전처리 파이프라인 =====
preprocess = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])


# ===== 모델 로드 =====
def load_model(model_path: str, num_classes: int = 10) -> nn.Module:
    """저장된 state_dict를 불러와서 추론 가능한 모델을 반환합니다."""
    model = SimpleClassifier(num_classes=num_classes)
    model.load_state_dict(
        torch.load(model_path, map_location="cpu", weights_only=True)
    )
    model.eval()
    return model


# ===== 추론 =====
# 클래스 이름 매핑
CLASS_NAMES = [str(i) for i in range(10)]   # MNIST: "0" ~ "9"


def predict(model: nn.Module, image_tensor: torch.Tensor) -> dict:
    """
    전처리된 이미지 텐서를 받아 추론 결과를 반환합니다.

    Args:
        model: 로드된 PyTorch 모델
        image_tensor: (1, 1, 28, 28) 형태의 텐서

    Returns:
        {
            "predicted_class": "7",
            "confidence": 0.98,
            "probabilities": {"0": 0.001, "1": 0.002, ...}
        }
    """
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = torch.softmax(output, dim=1)[0]

        predicted_idx = probabilities.argmax().item()
        confidence = probabilities[predicted_idx].item()

        prob_dict = {
            CLASS_NAMES[i]: round(probabilities[i].item(), 4)
            for i in range(len(CLASS_NAMES))
        }

    return {
        "predicted_class": CLASS_NAMES[predicted_idx],
        "confidence": round(confidence, 4),
        "probabilities": prob_dict,
    }



Writing app/model_utils.py


In [51]:
# 작성한 모듈이 정상 동작하는지 테스트합니다
import sys
sys.path.insert(0, ".")

from app.model_utils import load_model, predict, preprocess

# 모델 로드
model_for_api = load_model("models/mnist_state_dict.pth")

# 추론 테스트
result = predict(model_for_api, test_input)

print("추론 결과:")
print(f"  예측 클래스: {result['predicted_class']}")
print(f"  확신도:     {result['confidence']}")
print(f"  전체 확률:")
for cls, prob in result['probabilities'].items():
    bar = "█" * int(prob * 50)
    print(f"    {cls}: {prob:.4f} {bar}")

추론 결과:
  예측 클래스: 7
  확신도:     1.0
  전체 확률:
    0: 0.0000 
    1: 0.0000 
    2: 0.0000 
    3: 0.0000 
    4: 0.0000 
    5: 0.0000 
    6: 0.0000 
    7: 1.0000 ██████████████████████████████████████████████████
    8: 0.0000 
    9: 0.0000 


In [52]:
import os

def show_tree(path, prefix="", max_depth=2, current_depth=0):
    """프로젝트 폴더 구조를 트리 형태로 출력합니다."""
    if current_depth >= max_depth:
        return

    entries = sorted(os.listdir(path))
    

    for i, entry in enumerate(entries):
        full_path = os.path.join(path, entry)
        connector = "└── " if i == len(entries) - 1 else "├── "

        if os.path.isdir(full_path):
            print(f"{prefix}{connector}📁 {entry}/")
            extension = "    " if i == len(entries) - 1 else "│   "
            show_tree(full_path, prefix + extension, max_depth, current_depth + 1)
        else:
            size = os.path.getsize(full_path)
            if size > 1024:
                size_str = f"({size/1024:.1f} KB)"
            else:
                size_str = f"({size} B)"
            print(f"{prefix}{connector}{entry} {size_str}")

print("model-serving-course/")
show_tree(".")

model-serving-course/
├── 📁 .venv/
│   ├── 📁 Include/
│   ├── 📁 Lib/
│   ├── 📁 Scripts/
│   ├── 📁 etc/
│   ├── pyvenv.cfg (282 B)
│   └── 📁 share/
├── 📁 app/
│   ├── 📁 __pycache__/
│   └── model_utils.py (2.6 KB)
├── 📁 data/
│   └── 📁 MNIST/
├── 📁 frontend/
├── 📁 models/
│   ├── mnist_model.onnx (1649.1 KB)
│   ├── mnist_state_dict.pth (1650.5 KB)
│   ├── mnist_traced.pt (1674.2 KB)
│   ├── model.onnx (1649.1 KB)
│   ├── model_state_dict.pth (1650.5 KB)
│   └── model_traced.pt (1673.3 KB)
├── 📁 notebooks/
├── requirements.txt (298 B)
├── serving.ipynb (77.0 KB)
└── 📁 tests/
